# CS448 - Lab 6: Denoising

In this lab we will use denoising to clean up some noisy recordings. We will use the following sounds:

Speech in a room: ```room-speech.wav```

ATC: ```aircomm.wav```

Speech with wind: ```wind-speech.wav```



## Part 1. Cleaning up the sounds

Let’s start by cleaning up these sounds as best as we can. We will do a straightforward magnitude spectral subtraction. For all of these sounds there is only noise in the first few seconds of the recording so that you can learn a noise profile from there. Do the following:
Perform an STFT of the recordings

- Estimate the magnitude spectrum of the noise from the beginning of the recording
    - It’s up to you to figure out how many seconds to use (hint: look at the spectrogram)

- Perform spectral subtraction by subtracting that spectrum from the input’s magnitude STFT
    - Remember to clip any resulting negative values to zero
    - Try to find how much of the noise to subtract so that the output looks good

- Use the original signal’s phase to convert back to a time series.

Make a note of which examples sound the best and are easier to work with. Try to explain why.
At this point some of the outputs will exhibit “musical noise”. To minimize its effects apply a median filter on the denoised magnitude spectrogram to make it sound better (hint: ```scipy.signal.medfilt2```). How big should the median window be? Try different values and find which work best.


In [1]:
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
import scipy.signal as signal
# Make a sound player function that plays array "x" with a sample rate "rate", and labels it with "label"
def sound_player( x, rate=8000, label=''):
    from IPython.display import display, Audio, HTML
    display( HTML( 
    '<style> table, th, td {border: 0px; }</style> <table><tr><td>' + label + 
    '</td><td>' + Audio( x, rate=rate)._repr_html_()[3:] + '</td></tr></table>'
    ))


def stft( input_sound, dft_size, hop_size, window):
    """Compute the short-time Fourier transform of a sound.
    Args:
        input_sound: 1D numpy array containing the sound samples
        dft_size: Size of the DFT to compute (number of frequency bins)
        hop_size: Number of samples to advance between successive frames
        window: 1D numpy array containing the analysis window (length = dft_size)
    Returns:
        f: 2D numpy array (frequencies x time) containing the complex-valued STFT
    """
    input_sound_length = len(input_sound)
    num_frames = 1 + (input_sound_length - dft_size) // hop_size
    spectrogram = np.zeros((dft_size // 2 + 1, num_frames), dtype=np.complex64)
    for frame in range(num_frames):
        start = frame * hop_size
        end = start + dft_size
        segment = input_sound[start:end] * window
        spectrum = np.fft.rfft(segment, n=dft_size)
        spectrogram[:, frame] = spectrum

    # Return a complex-valued spectrogram (frequencies x time)    
    return spectrogram
    # # YOUR CODE HERE
    # raise NotImplementedError()

def istft( stft_output, dft_size, hop_size, window):
    """Compute the inverse short-time Fourier transform of a spectrogram.
    Args:
        stft_output: 2D numpy array (frequencies x time) containing the complex-valued STFT
        dft_size: Size of the DFT that was computed (number of frequency bins)
        hop_size: Number of samples advanced between successive frames
        window: 1D numpy array containing the synthesis window (length = dft_size)
    Returns:
        x: 1D numpy array containing the reconstructed sound samples
    """
    number_frames = stft_output.shape[1]
    output_length = (number_frames - 1) * hop_size + dft_size
    x = np.zeros(output_length)
    
    for frame in range(number_frames):
        start = frame * hop_size
        end = start + dft_size
        windowed_frame = np.fft.irfft(stft_output[:, frame], n=dft_size)
        x[start:end] += windowed_frame * window
    
    return x

dft_size=1024
hop_size=512




In [142]:
# load signals

# stft

# Plot the spectrograms of the original signals



In [143]:
# Define noise pattern areas (by eyeballing the spectrograms)

# Compute the noise pattern by averaging the magnitude of the spectrogram in the specified areas

# Set the over-subtraction factor alpha (you can experiment with different values)

# Perform spectral subtraction



In [144]:
# Perform median filtering on the magnitude of the spectrogram

# Check out the sound


In [145]:
# Apply median filtering to the magnitude of the spectrogram to remove musical noise
# Play around with the kernel size to see how it affects the results (try 3, 5, 7, etc.)

# STFT and show the spectrograms of the denoised signals (after spectral subtraction and median filtering)

# Reconstruct the denoised and median filtered signals using the inverse STFT

# check out the sound

## Part 2. Implement a Voice Activity Detector (VAD)


For the last sound we have an evolving noise profile, which causes problems since our noise description from the first two seconds isn’t accurate throughout. Because we’re lazy we want to automatically update the noise model and not to select it manually. To do so we need a VAD that lets us know when to gather noise statistics and when to denoise. Do the following:

- Take the square of the input waveform and lowpass filter it (a lot) to get an energy level over time
    - Experiment with the cutoff frequency so that you get a smooth energy-looking function
- Set a threshold over which we seem to have speech in the input
- Implement a real-time spectral subtraction denoiser
    - If an input frame is under threshold, it is noise
    - Keep track of the last few noise frames and their average amplitude will be your noise spectrum
    - If an input is over the threshold it is speech
    - Once you encounter speech perform spectral subtraction with the current noise estimate

In [147]:
# Set up the FIR low-pass filter parameters (a very drastic one)

# Apply the filter to the sqaured signal

# set the appropriate threshold

# Convert the thresholded VAD score curve into STFT to match the frame rates

# Adaptive spectral subtraction using the VAD output to update the noise estimate only in the non-speech frames

# set the buffer size for noise estimation (you can experiment with different values, e.g., 5, 10, 20, etc.)

# Loop through each frame of the spectrogram and apply spectral subtraction with adaptive noise estimation

# for i in range(number_of_frames):
#     if # non-speech frame, update the noise estimate buffer
      
#     else: # speech frame, perform spectral subtraction using the current noise estimate
      
# check out the spectrograms and filtered sound